[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adaptive-intelligent-robotics/QDax/blob/main/examples/pgame.ipynb)

# Optimizing with DCRL-ME in JAX

This notebook shows how to use QDax to find diverse and performing controllers in MDPs with [Descriptor-Conditioned Reinforcement Learning MAP-Elites (DCRL-ME)](https://arxiv.org/abs/2401.08632).
This algorithm extends and improves upon [Descriptor-Conditioned Gradients MAP-Elites (DCG-ME)](https://dl.acm.org/doi/abs/10.1145/3583131.3590503)
It can be run locally or on Google Colab. We recommend to use a GPU. This notebook will show:

- how to define the problem
- how to create the DCRL emitter
- how to create a Map-elites instance
- which functions must be defined before training
- how to launch a certain number of training steps
- how to visualize the results of the training process

In [ ]:
!git clone https://github.com/adaptive-intelligent-robotics/QDax.git

In [ ]:
%cd QDax

In [ ]:
%%writefile qdax/core/emitters/dcrl_me_sac_emitter.py

from dataclasses import dataclass
from typing import Callable, Optional, Tuple

import flax.linen as nn

from qdax.core.emitters.dcrl_sac_emitter import DCRLConfig, DCRLEmitter
from qdax.core.emitters.multi_emitter import MultiEmitter
from qdax.core.emitters.repertoire_selectors.selector import Selector
from qdax.core.emitters.standard_emitters import MixingEmitter
from qdax.custom_types import Params, RNGKey
from qdax.tasks.brax.v1.envs.base_env import QDEnv


@dataclass
class DCRLMEConfig:
    """Configuration for DCRL-MAP-Elites Algorithm"""

    ga_batch_size: int = 128
    dcrl_batch_size: int = 64
    ai_batch_size: int = 64
    lengthscale: float = 0.1

    # PG emitter
    critic_hidden_layer_size: Tuple[int, ...] = (256, 256)
    num_critic_training_steps: int = 3000
    num_pg_training_steps: int = 150
    batch_size: int = 100
    replay_buffer_size: int = 1_000_000
    discount: float = 0.99
    reward_scaling: float = 1.0
    critic_learning_rate: float = 3e-4
    actor_learning_rate: float = 3e-4
    policy_learning_rate: float = 1e-3
    alpha_learning_rate: float = 1e-3
    soft_tau_update: float = 0.005
    alpha_init: float = 1.0
    policy_delay: int = 1


class DCRLMEEmitter(MultiEmitter):
    def __init__(
        self,
        config: DCRLMEConfig,
        policy_network: nn.Module,
        actor_network: nn.Module,
        env: QDEnv,
        variation_fn: Callable[[Params, Params, RNGKey], Tuple[Params, RNGKey]],
        selector: Optional[Selector] = None,
    ) -> None:
        self._config = config
        self._env = env
        self._variation_fn = variation_fn

        dcrl_config = DCRLConfig(
            dcrl_batch_size=config.dcrl_batch_size,
            ai_batch_size=config.ai_batch_size,
            lengthscale=config.lengthscale,
            critic_hidden_layer_size=config.critic_hidden_layer_size,
            num_critic_training_steps=config.num_critic_training_steps,
            num_pg_training_steps=config.num_pg_training_steps,
            batch_size=config.batch_size,
            replay_buffer_size=config.replay_buffer_size,
            discount=config.discount,
            reward_scaling=config.reward_scaling,
            critic_learning_rate=config.critic_learning_rate,
            actor_learning_rate=config.actor_learning_rate,
            policy_learning_rate=config.policy_learning_rate,
            alpha_learning_rate=config.alpha_learning_rate,
            soft_tau_update=config.soft_tau_update,
            policy_delay=config.policy_delay,
        )

        # define the quality emitter
        dcrl_emitter = DCRLEmitter(
            config=dcrl_config,
            policy_network=policy_network,
            actor_network=actor_network,
            env=env,
            selector=selector,
        )

        # define the GA emitter
        ga_emitter = MixingEmitter(
            mutation_fn=lambda x, r: (x, r),
            variation_fn=variation_fn,
            variation_percentage=1.0,
            batch_size=config.ga_batch_size,
            selector=selector,
        )

        super().__init__(emitters=(dcrl_emitter, ga_emitter))


In [ ]:
%%writefile qdax/core/emitters/dcrl_sac_emitter.py

"""Implements the DCRL Emitter from DCRL-MAP-Elites algorithm
in JAX for Brax environments.
"""

from dataclasses import dataclass
from functools import partial
from typing import Any, Optional, Tuple

import flax.linen as nn
import jax
import optax
from jax import numpy as jnp
from brax.training.distribution import NormalTanhDistribution

from qdax.core.containers.mapelites_repertoire import MapElitesRepertoire
from qdax.core.containers.repertoire import Repertoire
from qdax.core.emitters.emitter import Emitter, EmitterState
from qdax.core.emitters.repertoire_selectors.selector import Selector
from qdax.core.neuroevolution.buffers.buffer import DCRLTransition, ReplayBuffer
from qdax.core.neuroevolution.losses.sac_loss import make_sac_loss_dc_fn
from qdax.core.neuroevolution.networks.networks import QModuleDC
from qdax.custom_types import Descriptor, ExtraScores, Fitness, Genotype, Params, RNGKey
from qdax.tasks.brax.v1.envs.base_env import QDEnv


@dataclass
class DCRLConfig:
    """Configuration for DCRL Emitter"""

    dcrl_batch_size: int = 64
    ai_batch_size: int = 64
    lengthscale: float = 0.1

    critic_hidden_layer_size: Tuple[int, ...] = (256, 256)
    num_critic_training_steps: int = 3000
    num_pg_training_steps: int = 150
    batch_size: int = 100
    replay_buffer_size: int = 1_000_000
    discount: float = 0.99
    reward_scaling: float = 1.0
    critic_learning_rate: float = 3e-4
    actor_learning_rate: float = 3e-4
    policy_learning_rate: float = 1e-3
    alpha_learning_rate: float = 3e-4
    soft_tau_update: float = 0.005
    alpha_init: float = 1.0
    policy_delay: int = 1


class DCRLEmitterState(EmitterState):
    """Contains training state for the learner."""

    critic_params: Params
    critic_opt_state: optax.OptState
    actor_params: Params
    actor_opt_state: optax.OptState
    target_critic_params: Params
    alpha_params: Params
    alpha_opt_state: optax.OptState
    replay_buffer: ReplayBuffer
    key: RNGKey


class DCRLEmitter(Emitter):
    """
    A descriptor-conditioned reinforcement learning emitter used to implement
    DCRL-MAP-Elites algorithm.
    """

    def __init__(
        self,
        config: DCRLConfig,
        policy_network: nn.Module,
        actor_network: nn.Module,
        env: QDEnv,
        selector: Optional[Selector] = None,
    ) -> None:
        self._config = config
        self._env = env
        self._selector = selector
        self._actor_network = actor_network
        self._actor_critic_iterations = int(
            config.num_critic_training_steps / config.policy_delay
        )  # actor and critic training are packed into a single function

        # Init Critics
        critic_network = QModuleDC(
            n_critics=2, hidden_layer_sizes=self._config.critic_hidden_layer_size
        )
        self._critic_network = critic_network

        # Set up the losses and optimizers - return the opt states
        (
            self._policy_loss_fn,
            self._actor_loss_fn,
            self._critic_loss_fn,
            self._alpha_loss_fn,
        ) = make_sac_loss_dc_fn(
            policy_fn=policy_network.apply,
            actor_fn=actor_network.apply,
            critic_fn=critic_network.apply,
            parametric_action_distribution=NormalTanhDistribution(event_size=self._env.action_size),
            reward_scaling=self._config.reward_scaling,
            discount=self._config.discount,
            action_size=self._env.action_size,
            lengthscale=self._config.lengthscale
        )

        # Init optimizers
        self._actor_optimizer = optax.adam(
            learning_rate=self._config.actor_learning_rate
        )
        self._critic_optimizer = optax.adam(
            learning_rate=self._config.critic_learning_rate
        )
        self._policies_optimizer = optax.adam(
            learning_rate=self._config.policy_learning_rate
        )
        self._alpha_optimizer = optax.adam(
            learning_rate=self._config.alpha_learning_rate
        )

    @property
    def batch_size(self) -> int:
        """
        Returns:
            the batch size emitted by the emitter.
        """
        return self._config.dcrl_batch_size + self._config.ai_batch_size

    @property
    def use_all_data(self) -> bool:
        """Whether to use all data or not when used along other emitters"""
        return True

    def init(
        self,
        key: RNGKey,
        repertoire: Repertoire,
        genotypes: Genotype,
        fitnesses: Fitness,
        descriptors: Descriptor,
        extra_scores: ExtraScores,
    ) -> DCRLEmitterState:
        """Initializes the emitter state.

        Args:
            genotypes: The initial population.
            key: A random key.

        Returns:
            The initial state of the PGAMEEmitter
        """

        observation_size = jax.tree.leaves(genotypes)[1].shape[1]
        descriptor_size = self._env.descriptor_length
        action_size = self._env.action_size

        # Initialise critic, greedy actor and population
        key, subkey = jax.random.split(key)
        fake_obs = jnp.zeros(shape=(observation_size,))
        fake_desc = jnp.zeros(shape=(descriptor_size,))
        fake_action = jnp.zeros(shape=(action_size,))

        critic_params = self._critic_network.init(
            subkey, obs=fake_obs, actions=fake_action, desc=fake_desc
        )
        target_critic_params = jax.tree.map(lambda x: x, critic_params)

        key, subkey = jax.random.split(key)
        actor_params = self._actor_network.init(subkey, obs=fake_obs, desc=fake_desc)

        # Prepare init optimizer states
        critic_opt_state = self._critic_optimizer.init(critic_params)
        actor_opt_state = self._actor_optimizer.init(actor_params)


        # Initialize Alpha
        log_alpha = jnp.asarray(jnp.log(self._config.alpha_init), dtype=jnp.float32)
        alpha_opt_state = self._alpha_optimizer.init(log_alpha)


        # Initialize replay buffer
        dummy_transition = DCRLTransition.init_dummy(
            observation_dim=self._env.observation_size,
            action_dim=action_size,
            descriptor_dim=descriptor_size,
        )

        replay_buffer = ReplayBuffer.init(
            buffer_size=self._config.replay_buffer_size, transition=dummy_transition
        )

        assert "transitions" in extra_scores.keys(), "Missing transitions or wrong key"
        transitions = extra_scores["transitions"]
        episode_length = transitions.obs.shape[1]

        desc = jnp.repeat(descriptors[:, None, :], episode_length, axis=1)
        desc_normalized = jax.vmap(jax.vmap(self._normalize_desc))(desc)

        transitions = transitions.replace(
            desc=desc_normalized, desc_prime=desc_normalized
        )
        replay_buffer = replay_buffer.insert(transitions)

        # Initial training state
        key, subkey = jax.random.split(key)
        emitter_state = DCRLEmitterState(
            critic_params=critic_params,
            critic_opt_state=critic_opt_state,
            actor_params=actor_params,
            actor_opt_state=actor_opt_state,
            target_critic_params=target_critic_params,
            alpha_params=log_alpha,
            alpha_opt_state=alpha_opt_state,
            replay_buffer=replay_buffer,
            key=subkey,
        )

        return emitter_state

    def _similarity(self, descs_1: Descriptor, descs_2: Descriptor) -> jnp.array:
        """Compute the similarity between two batches of descriptors.
        Args:
            descs_1: batch of descriptors.
            descs_2: batch of descriptors.
        Returns:
            batch of similarity measures.
        """
        return jnp.exp(
            -jnp.linalg.norm(descs_1 - descs_2, axis=-1) / self._config.lengthscale
        )

    def _normalize_desc(self, desc: Descriptor) -> Descriptor:
        return (
            2
            * (desc - self._env.descriptor_limits[0])
            / (self._env.descriptor_limits[1] - self._env.descriptor_limits[0])
            - 1
        )

    def _unnormalize_desc(self, desc_normalized: Descriptor) -> Descriptor:
        return 0.5 * (
            self._env.descriptor_limits[1] - self._env.descriptor_limits[0]
        ) * desc_normalized + 0.5 * (
            self._env.descriptor_limits[1] + self._env.descriptor_limits[0]
        )

    def _compute_equivalent_kernel_bias_with_desc(
        self, actor_dc_params: Params, desc: Descriptor
    ) -> Tuple[Params, Params]:
        """
        Compute the equivalent bias of the first layer of the actor network
        given a descriptor.
        """
        # Extract kernel and bias of the first layer
        kernel = actor_dc_params["params"]["Dense_0"]["kernel"]
        bias = actor_dc_params["params"]["Dense_0"]["bias"]

        # Compute the equivalent bias
        equivalent_kernel = kernel[: -desc.shape[0], :]
        equivalent_bias = bias + jnp.dot(desc, kernel[-desc.shape[0] :])

        return equivalent_kernel, equivalent_bias

    def _compute_equivalent_params_with_desc(
        self, actor_dc_params: Params, desc: Descriptor
    ) -> Params:
        desc_normalized = self._normalize_desc(desc)
        (
            equivalent_kernel,
            equivalent_bias,
        ) = self._compute_equivalent_kernel_bias_with_desc(
            actor_dc_params, desc_normalized
        )
        actor_dc_params["params"]["Dense_0"]["kernel"] = equivalent_kernel
        actor_dc_params["params"]["Dense_0"]["bias"] = equivalent_bias
        return actor_dc_params

    def emit(  # type: ignore
        self,
        repertoire: MapElitesRepertoire,
        emitter_state: DCRLEmitterState,
        key: RNGKey,
    ) -> Tuple[Genotype, ExtraScores]:
        """Do a step of policy-gradient and actor-injection emission.

        Args:
            repertoire: the current repertoire of genotypes
            emitter_state: the state of the emitter used
            key: a random key

        Returns:
            A batch of offspring with intended descs as extra information
        """
        # PG emitter
        subkey1, subkey2 = jax.random.split(key)
        sub_repertoire = repertoire.select(
            subkey1, self._config.dcrl_batch_size, selector=self._selector
        )
        parents_pg = sub_repertoire.genotypes
        descs_pg = sub_repertoire.descriptors
        genotypes_pg = self.emit_pg(emitter_state, parents_pg, descs_pg)

        # Actor injection emitter
        descs_ai = repertoire.select(
            subkey2, self._config.ai_batch_size, selector=self._selector
        ).descriptors
        descs_ai = descs_ai.reshape(descs_ai.shape[0], self._env.descriptor_length)
        genotypes_ai = self.emit_ai(emitter_state, descs_ai)

        # Concatenate PG and AI genotypes
        genotypes = jax.tree.map(
            lambda x, y: jnp.concatenate((x, y), axis=0), genotypes_pg, genotypes_ai
        )

        return (
            genotypes,
            {"desc_prime": jnp.concatenate([descs_pg, descs_ai], axis=0)},
        )

    def emit_pg(
        self,
        emitter_state: DCRLEmitterState,
        parents: Genotype,
        descs: Descriptor,
    ) -> Genotype:
        """Emit the offsprings generated through pg mutation.

        Args:
            emitter_state: current emitter state, contains critic and
                replay buffer.
            parents: the parents selected to be applied gradients in order
                to mutate towards better performance.
            descs: the descriptors of the parents.

        Returns:
            A new set of offsprings.
        """
        # normalize the intended descriptors and prepare them for concatenation
        descs_prime_normalized = jnp.tile(
            jax.vmap(self._normalize_desc)(descs), self._config.batch_size
        ).reshape(self._config.dcrl_batch_size, self._config.batch_size, -1)

        # create a batch of policy optimizer states
        policy_opt_states = jax.vmap(self._policies_optimizer.init)(parents)

        # prepare the batched policy update function with vmapping
        batched_policy_update_fn = jax.vmap(
            partial(self._update_policy, critic_params=emitter_state.critic_params, alpha_params=emitter_state.alpha_params),
            in_axes=(0, 0, 0, None, 0),
        )

        def scan_update_policies(
            carry: Tuple[Params, optax.OptState, jnp.ndarray, RNGKey],
            _: None,
        ) -> Tuple[Tuple[Params, optax.OptState, jnp.ndarray, RNGKey], Any]:

            (policy_params, policy_opt_state, desc_prime, key) = carry
            key, subkey = jax.random.split(key)

            # sample a mini-batch of data from the replay-buffer
            transitions = emitter_state.replay_buffer.sample(
                subkey, self._config.batch_size
            )

            subkeys = jax.random.split(key, self._config.dcrl_batch_size)
            
            batch_carry = batched_policy_update_fn(
                policy_params, policy_opt_state, desc_prime, transitions, subkeys
            )

            new_policy_params, new_policy_opt_states = batch_carry

            return (new_policy_params, new_policy_opt_states, desc_prime, key), ()

        (
            final_policy_params,
            policy_opt_state,
            desc_prime,
            key,
        ), _ = jax.lax.scan(
            scan_update_policies,
            (parents, policy_opt_states, descs_prime_normalized, emitter_state.key),
            length=self._config.num_pg_training_steps,
        )

        return final_policy_params

    def emit_ai(self, emitter_state: DCRLEmitterState, descs: Descriptor) -> Genotype:
        """Emit the offsprings generated through pg mutation.

        Args:
            emitter_state: current emitter state, contains critic and
                replay buffer.
            parents: the parents selected to be applied gradients in order
                to mutate towards better performance.
            descs: the descriptors of the parents.

        Returns:
            A new set of offsprings.
        """
        offsprings = jax.vmap(
            self._compute_equivalent_params_with_desc, in_axes=(None, 0)
        )(emitter_state.actor_params, descs)

        return offsprings

    def emit_actor(self, emitter_state: DCRLEmitterState) -> Genotype:
        """Emit the greedy actor.

        Simply needs to be retrieved from the emitter state.

        Args:
            emitter_state: the current emitter state, it stores the
                greedy actor.

        Returns:
            The parameters of the actor.
        """
        return emitter_state.actor_params

    def state_update(  # type: ignore
        self,
        emitter_state: DCRLEmitterState,
        repertoire: Repertoire,
        genotypes: Genotype,
        fitnesses: Fitness,
        descriptors: Descriptor,
        extra_scores: ExtraScores,
    ) -> DCRLEmitterState:
        """This function gives an opportunity to update the emitter state
        after the genotypes have been scored.

        Here it is used to fill the Replay Buffer with the transitions
        from the scoring of the genotypes, and then the training of the
        critic/actor happens. Hence the params of critic/actor are updated,
        as well as their optimizer states.

        Args:
            emitter_state: current emitter state.
            repertoire: the current genotypes repertoire
            genotypes: unused here - but compulsory in the signature.
            fitnesses: unused here - but compulsory in the signature.
            descriptors: unused here - but compulsory in the signature.
            extra_scores: extra information coming from the scoring function,
                this contains the transitions added to the replay buffer.

        Returns:
            New emitter state where the replay buffer has been filled with
            the new experienced transitions.
        """
        assert "transitions" in extra_scores.keys(), "Missing transitions or wrong key"
        transitions = extra_scores["transitions"]
        episode_length = transitions.obs.shape[1]

        desc_prime = jnp.concatenate(
            [
                extra_scores["desc_prime"],
                descriptors[
                    self._config.dcrl_batch_size + self._config.ai_batch_size :
                ],
            ],
            axis=0,
        )
        desc_prime = jnp.repeat(desc_prime[:, jnp.newaxis, :], episode_length, axis=1)
        desc = jnp.repeat(descriptors[:, jnp.newaxis, :], episode_length, axis=1)

        desc_prime_normalized = jax.vmap(jax.vmap(self._normalize_desc))(desc_prime)
        desc_normalized = jax.vmap(jax.vmap(self._normalize_desc))(desc)
        transitions = transitions.replace(
            desc=desc_normalized, desc_prime=desc_prime_normalized
        )

        # Add transitions to replay buffer
        emitter_state = emitter_state.replace(
            replay_buffer=emitter_state.replay_buffer.insert(transitions)
        )
        # Conduct Actor-Critic training
        final_emitter_state, _ = jax.lax.scan(
            self._scan_actor_critic_training,
            emitter_state,
            length=self._actor_critic_iterations,
        )

        return final_emitter_state  # type: ignore

    def _scan_update_critic(
        self,
        carry: Tuple[Params, Params, optax.OptState, Params, RNGKey],
        transitions: DCRLTransition,
    ) -> Tuple[Tuple[Params, Params, optax.OptState, Params, RNGKey], Any]:
        """A scan-ready function to update the critic network parameters
        with one gradient step.

        Args:
            carry: packed carry containing critic parameters, target critic
                parameters, critic optimiser state, target actor parameters,
                and the random key.
            transitions: a mini-batch of DCRLtransitions for gradient computation

        Returns:
            new_carry: new carry containing updated parameters (target actor
                parameters is unchanged).
            empty tuple: compulsory signature for jax.lax.scan
        """
        # unpack the carry
        (
            critic_params,
            target_critic_params,
            critic_opt_state,
            actor_params,
            alpha,
            key,
        ) = carry

        # compute critic gradients
        key, subkey = jax.random.split(key)
        critic_gradient = jax.grad(self._critic_loss_fn)(
            critic_params,
            actor_params,
            target_critic_params,
            jnp.exp(alpha),
            transitions,
            subkey,
        )
        critic_updates, new_critic_opt_state = self._critic_optimizer.update(
            critic_gradient, critic_opt_state
        )

        # update critic
        new_critic_params = optax.apply_updates(critic_params, critic_updates)

        # Soft update of target critic network
        new_target_critic_params = jax.tree.map(
            lambda x, y: (1.0 - self._config.soft_tau_update) * x
            + self._config.soft_tau_update * y,
            target_critic_params,
            new_critic_params,
        )
        # pack into new carry
        new_carry = (
            new_critic_params,
            new_target_critic_params,
            new_critic_opt_state,
            actor_params,
            alpha,
            key,
        )

        return new_carry, ()

    def _update_actor(
        self,
        actor_params: Params,
        actor_opt_state: optax.OptState,
        critic_params: Params,
        transitions: DCRLTransition,
        alpha: Params,
        key: RNGKey,
    ) -> Tuple[optax.OptState, Params, Params]:
        """Function to update the DC-actor with one Policy-Gradient step.

        Args:
            actor_params: neural network parameters of the DC-actor.
            actor_opt_state: optimiser state of the actor.
            critic_params: the parameters of the critic networks which plays
                as a differentiable target.
            transitions: a mini-batch of DCRLtransitions.

        Returns:
            new_actor_params: new actor parameters after taking the PG step
            new_actor_opt_state: updated optimiser state
        """

        # Update greedy actor
        policy_gradient = jax.grad(self._actor_loss_fn)(
            actor_params,
            critic_params,
            jnp.exp(alpha),
            transitions,
            key,
        )
        (
            policy_updates,
            new_actor_opt_state,
        ) = self._actor_optimizer.update(policy_gradient, actor_opt_state)
        new_actor_params = optax.apply_updates(actor_params, policy_updates)

        return new_actor_params, new_actor_opt_state
    

    def _update_alpha(
        self,
        actor_params: Params,
        log_alpha: Params,
        alpha_opt_state: optax.OptState,
        transitions: DCRLTransition,
        key: RNGKey,
    ) -> Tuple[optax.OptState, Params, Params]:
        """Function to update the DC-actor with one Policy-Gradient step.

        Args:
            actor_params: neural network parameters of the DC-actor.
            actor_opt_state: optimiser state of the actor.
            critic_params: the parameters of the critic networks which plays
                as a differentiable target.
            transitions: a mini-batch of DCRLtransitions.

        Returns:
            new_actor_params: new actor parameters after taking the PG step
            new_actor_opt_state: updated optimiser state
        """

        # Update greedy actor
        alpha_gradient = jax.grad(self._alpha_loss_fn)(
            log_alpha,
            actor_params,
            transitions,
            key,
        )
        (
            alpha_updates,
            new_alpha_opt_state,
        ) = self._alpha_optimizer.update(alpha_gradient, alpha_opt_state)
        new_log_alpha = optax.apply_updates(log_alpha, alpha_updates)

        return new_log_alpha, new_alpha_opt_state

    def _scan_actor_critic_training(
        self, carry: DCRLEmitterState, _: None
    ) -> Tuple[DCRLEmitterState, Tuple]:
        """
        Perform a few (policy delay) steps of critic followed by one step of
        actor training, all packed into a single scan-ready function.
        Transition data are sampled step-by-step to promote memory efficiency.

        Args:
            carry: emitter state
            _: None

        Returns:
            new_emitter_state: new emitter state containing updated network
                parameters as the new carry.
            empty tuple: compulsory signature for jax.lax.scan
        """

        emitter_state = carry
        key, subkey = jax.random.split(emitter_state.key)

        # sample transitions
        transitions = emitter_state.replay_buffer.sample(
            subkey, self._config.batch_size * (self._config.policy_delay + 1)
        )
        transitions = jax.tree.map(
            lambda x: jnp.reshape(
                x,
                (
                    self._config.policy_delay + 1,
                    self._config.batch_size,
                    *x.shape[1:],
                ),
            ),
            transitions,
        )

        # rescale rewards by descriptor similarity
        transitions = transitions.replace(
            rewards=self._similarity(transitions.desc, transitions.desc_prime)
            * transitions.rewards
        )
        # split the transitions for critic and actor
        critic_data = jax.tree.map(lambda x: x[:-1], transitions)
        actor_data = jax.tree.map(lambda x: x[-1], transitions)

        # scan training critics
        (
            new_critic_params,
            new_target_critic_params,
            new_critic_opt_state,
            actor_params,
            alpha_params,
            key,
        ), () = jax.lax.scan(
            self._scan_update_critic,
            (
                emitter_state.critic_params,
                emitter_state.target_critic_params,
                emitter_state.critic_opt_state,
                emitter_state.actor_params,
                emitter_state.alpha_params,
                key,

                
            ),
            critic_data,
        )

        # train actor with one gradient step
        (new_actor_params, new_actor_opt_state) = (
            self._update_actor(
                emitter_state.actor_params,
                emitter_state.actor_opt_state,
                new_critic_params,
                actor_data,
                emitter_state.alpha_params,
                key,
            )
        )

        # train alpha with one gradient step
        (new_alpha_params, new_alpha_opt_state) = (
            self._update_alpha(
                emitter_state.actor_params,
                emitter_state.alpha_params,
                emitter_state.alpha_opt_state,
                actor_data,
                key,
            )
        )

        new_emitter_state = emitter_state.replace(
            critic_params=new_critic_params,
            critic_opt_state=new_critic_opt_state,
            actor_params=new_actor_params,
            actor_opt_state=new_actor_opt_state,
            target_critic_params=new_target_critic_params,
            alpha_params=new_alpha_params,
            alpha_opt_state=new_alpha_opt_state,
            key=key,
        )

        return new_emitter_state, ()

    def _update_policy(
        self,
        policy_params: Params,
        policy_opt_state: optax.OptState,
        desc_prime: Descriptor,
        transitions: DCRLTransition,
        key: RNGKey,
        critic_params: Params,
        alpha_params: jnp.ndarray,
    ) -> Tuple[Params, optax.OptState]:
        """Perform one step of PG update on the off-spring policy.
        This function is vmapped to mutate the entire batch of off-springs
        in parallel.

        Args:
            policy_params: the parameters of the policy.
            policy_opt_state: the optimiser state of the policy.
            desc_prime: the normalised intended descriptor of the policy.
            transitions: a mini-batch of transitions for gradient computation
            critic_params: the parameters of the critic networks serving as
                a differentiable target. This is fixed in each iteration.

        Returns:
            new_policy_params: new policy parameters
            new_policy_opt_state: updated optimiser state
        """

        # Compute gradient
        policy_gradient = jax.grad(self._policy_loss_fn)(
            policy_params,
            critic_params,
            desc_prime,
            alpha_params,
            transitions,
            key,
        )
        # Update the policy
        (
            policy_updates,
            new_policy_opt_state,
        ) = self._policies_optimizer.update(policy_gradient, policy_opt_state)
        new_policy_params = optax.apply_updates(policy_params, policy_updates)

        return new_policy_params, new_policy_opt_state


In [ ]:
%%writefile qdax/core/neuroevolution/losses/sac_loss.py

import functools
from typing import Callable, Tuple

import jax
import jax.numpy as jnp
from brax.training.distribution import ParametricDistribution

from qdax.core.neuroevolution.buffers.buffer import Transition, DCRLTransition
from qdax.custom_types import Action, Observation, Descriptor, Params, RNGKey


def make_sac_loss_fn(
    policy_fn: Callable[[Params, Observation], jnp.ndarray],
    critic_fn: Callable[[Params, Observation, Action], jnp.ndarray],
    parametric_action_distribution: ParametricDistribution,
    reward_scaling: float,
    discount: float,
    action_size: int,
) -> Tuple[
    Callable[[jnp.ndarray, Params, Transition, RNGKey], jnp.ndarray],
    Callable[[Params, Params, jnp.ndarray, Transition, RNGKey], jnp.ndarray],
    Callable[[Params, Params, Params, Transition, RNGKey], jnp.ndarray],
]:
    """Creates the loss used in SAC.

    Args:
        policy_fn: the apply function of the policy
        critic_fn: the apply function of the critic
        parametric_action_distribution: the distribution over actions
        reward_scaling: a multiplicative factor to the reward
        discount: the discount factor
        action_size: the size of the environment's action space

    Returns:
        the loss of the entropy parameter auto-tuning
        the loss of the policy
        the loss of the critic
    """

    _policy_loss_fn = functools.partial(
        sac_policy_loss_fn,
        policy_fn=policy_fn,
        critic_fn=critic_fn,
        parametric_action_distribution=parametric_action_distribution,
    )

    _critic_loss_fn = functools.partial(
        sac_critic_loss_fn,
        policy_fn=policy_fn,
        critic_fn=critic_fn,
        parametric_action_distribution=parametric_action_distribution,
        reward_scaling=reward_scaling,
        discount=discount,
    )

    _alpha_loss_fn = functools.partial(
        sac_alpha_loss_fn,
        policy_fn=policy_fn,
        parametric_action_distribution=parametric_action_distribution,
        action_size=action_size,
    )

    return _alpha_loss_fn, _policy_loss_fn, _critic_loss_fn








def make_sac_loss_dc_fn(
    policy_fn: Callable[[Params, Observation], jnp.ndarray],
    actor_fn: Callable[[Params, Observation, Descriptor], jnp.ndarray],
    critic_fn: Callable[[Params, Observation, Action, Descriptor], jnp.ndarray],
    parametric_action_distribution: ParametricDistribution,
    reward_scaling: float,
    discount: float,
    action_size: int,
    lengthscale: float,
):
    """Creates the loss functions for SAC.
    Args:
        policy_fn: forward pass through the neural network defining the policy.
        actor_fn: forward pass through the neural network defining the
            descriptor-conditioned policy.
        critic_fn: forward pass through the neural network defining the
            descriptor-conditioned critic.
        reward_scaling: value to multiply the reward given by the environment.
        discount: discount factor.
        noise_clip: value that clips the noise to avoid extreme values.
        policy_noise: noise applied to smooth the bootstrapping.
    Returns:
        Return the loss functions used to train the policy and the critic in TD3.
    """

    def similarity(descs_1: Descriptor, descs_2: Descriptor) -> jnp.array:
        """Compute the similarity between two batches of descriptors.
        Args:
            descs_1: batch of descriptors.
            descs_2: batch of descriptors.
        Returns:
            batch of similarity measures.
        """
        return jnp.exp(
            -jnp.linalg.norm(descs_1 - descs_2, axis=-1) / (lengthscale * 10)
        )

    def _policy_loss_fn(
        policy_params: Params,
        critic_params: Params,
        desc_prime: Descriptor,
        alpha: Params,
        transitions: DCRLTransition,
        key: RNGKey,
    ) -> jnp.ndarray:
        """Policy loss function for SAC agent"""
        dist_params = policy_fn(policy_params, transitions.obs)
        action = parametric_action_distribution.sample_no_postprocessing(dist_params, key)
        log_prob = parametric_action_distribution.log_prob(dist_params, action)
        action = parametric_action_distribution.postprocess(action)
        q_action = critic_fn(critic_params, transitions.obs, action, desc_prime)
        min_q = jnp.min(q_action, axis=-1)

        policy_loss = alpha * log_prob - min_q
        # policy_loss = - min_q

        return jnp.mean(policy_loss)

    def _actor_loss_fn(
        actor_params: Params,
        critic_params: Params,
        alpha: Params,
        transitions: DCRLTransition,
        key: RNGKey,
    ) -> jnp.ndarray:
        """Descriptor-conditioned policy loss function for SAC agent"""
        dist_params = actor_fn(actor_params, transitions.obs, transitions.desc_prime)
        action = parametric_action_distribution.sample_no_postprocessing(dist_params, key)
        log_prob = parametric_action_distribution.log_prob(dist_params, action)
        action = parametric_action_distribution.postprocess(action)
        q_action = critic_fn(critic_params, transitions.obs, action, transitions.desc_prime)
        min_q = jnp.min(q_action, axis=-1)
        actor_loss = alpha * log_prob - min_q
        # actor_loss = - min_q

        return jnp.mean(actor_loss)

    def _critic_loss_fn(
        critic_params: Params,
        policy_params: Params,
        target_critic_params: Params,
        alpha: Params,
        transitions: DCRLTransition,
        key: RNGKey,
    ) -> jnp.ndarray:
        """Descriptor-conditioned critic loss function for SAC agent"""
        q_old_action = critic_fn(critic_params, transitions.obs, transitions.actions, transitions.desc_prime)
        next_dist_params = actor_fn(policy_params, transitions.next_obs, transitions.desc_prime)
        next_action = parametric_action_distribution.sample_no_postprocessing(
            next_dist_params, key
        )
        next_log_prob = parametric_action_distribution.log_prob(
            next_dist_params, next_action
        )
        next_action = parametric_action_distribution.postprocess(next_action)
        next_q = critic_fn(target_critic_params, transitions.next_obs, next_action, transitions.desc_prime)

        next_v = jnp.min(next_q, axis=-1) - alpha * next_log_prob

        target_q = jax.lax.stop_gradient(
            transitions.rewards * reward_scaling
            + (1.0 - transitions.dones) * discount * next_v
        )

        q_error = q_old_action - jnp.expand_dims(target_q, -1)
        q_error *= jnp.expand_dims(1 - transitions.truncations, -1)
        q_loss = 0.5 * jnp.mean(jnp.square(q_error))

        return q_loss
    
    def _alpha_loss_fn(
        log_alpha: Params,
        actor_params: Params,
        transitions: DCRLTransition,
        key: RNGKey,
    ) -> jnp.ndarray:
        """Descriptor-conditioned alpha loss function for SAC agent"""

        target_entropy = -0.5 * action_size

        dist_params = actor_fn(actor_params, transitions.obs, transitions.desc_prime)
        action = parametric_action_distribution.sample_no_postprocessing(dist_params, key)
        log_prob = parametric_action_distribution.log_prob(dist_params, action)
        alpha = jnp.exp(log_alpha)
        alpha_loss = alpha * jax.lax.stop_gradient(-log_prob - target_entropy)

        loss = jnp.mean(alpha_loss)
        return loss

    return _policy_loss_fn, _actor_loss_fn, _critic_loss_fn, _alpha_loss_fn



def sac_policy_loss_fn(
    policy_params: Params,
    policy_fn: Callable[[Params, Observation], jnp.ndarray],
    critic_fn: Callable[[Params, Observation, Action], jnp.ndarray],
    parametric_action_distribution: ParametricDistribution,
    critic_params: Params,
    alpha: jnp.ndarray,
    transitions: Transition,
    key: RNGKey,
) -> jnp.ndarray:
    """
    Creates the policy loss used in SAC.

    Args:
        policy_params: parameters of the policy
        policy_fn: the apply function of the policy
        critic_fn: the apply function of the critic
        parametric_action_distribution: the distribution over actions
        critic_params: parameters of the critic
        alpha: entropy coefficient value
        transitions: transitions collected by the agent
        key: random key

    Returns:
        the loss of the policy
    """

    dist_params = policy_fn(policy_params, transitions.obs)
    action = parametric_action_distribution.sample_no_postprocessing(dist_params, key)
    log_prob = parametric_action_distribution.log_prob(dist_params, action)
    action = parametric_action_distribution.postprocess(action)
    q_action = critic_fn(critic_params, transitions.obs, action)
    min_q = jnp.min(q_action, axis=-1)
    actor_loss = alpha * log_prob - min_q

    return jnp.mean(actor_loss)


def sac_critic_loss_fn(
    critic_params: Params,
    policy_fn: Callable[[Params, Observation], jnp.ndarray],
    critic_fn: Callable[[Params, Observation, Action], jnp.ndarray],
    parametric_action_distribution: ParametricDistribution,
    reward_scaling: float,
    discount: float,
    policy_params: Params,
    target_critic_params: Params,
    alpha: jnp.ndarray,
    transitions: Transition,
    key: RNGKey,
) -> jnp.ndarray:
    """
    Creates the critic loss used in SAC.

    Args:
        critic_params: parameters of the critic
        policy_fn: the apply function of the policy
        critic_fn: the apply function of the critic
        parametric_action_distribution: the distribution over actions
        policy_params: parameters of the policy
        target_critic_params: parameters of the target critic
        alpha: entropy coefficient value
        transitions: transitions collected by the agent
        key: random key
        reward_scaling: a multiplicative factor to the reward
        discount: the discount factor

    Returns:
        the loss of the critic
    """

    q_old_action = critic_fn(critic_params, transitions.obs, transitions.actions)
    next_dist_params = policy_fn(policy_params, transitions.next_obs)
    next_action = parametric_action_distribution.sample_no_postprocessing(
        next_dist_params, key
    )
    next_log_prob = parametric_action_distribution.log_prob(
        next_dist_params, next_action
    )
    next_action = parametric_action_distribution.postprocess(next_action)
    next_q = critic_fn(target_critic_params, transitions.next_obs, next_action)

    next_v = jnp.min(next_q, axis=-1) - alpha * next_log_prob

    target_q = jax.lax.stop_gradient(
        transitions.rewards * reward_scaling
        + (1.0 - transitions.dones) * discount * next_v
    )

    q_error = q_old_action - jnp.expand_dims(target_q, -1)
    q_error *= jnp.expand_dims(1 - transitions.truncations, -1)
    q_loss = 0.5 * jnp.mean(jnp.square(q_error))

    return q_loss


def sac_alpha_loss_fn(
    log_alpha: jnp.ndarray,
    policy_fn: Callable[[Params, Observation], jnp.ndarray],
    parametric_action_distribution: ParametricDistribution,
    action_size: int,
    policy_params: Params,
    transitions: Transition,
    key: RNGKey,
) -> jnp.ndarray:
    """
    Creates the alpha loss used in SAC.
    Eq 18 from https://arxiv.org/pdf/1812.05905.pdf.

    Args:
        log_alpha: entropy coefficient log value
        policy_fn: the apply function of the policy
        parametric_action_distribution: the distribution over actions
        policy_params: parameters of the policy
        transitions: transitions collected by the agent
        key: random key
        action_size: the size of the environment's action space

    Returns:
        the loss of the entropy parameter auto-tuning
    """

    target_entropy = -0.5 * action_size

    dist_params = policy_fn(policy_params, transitions.obs)
    action = parametric_action_distribution.sample_no_postprocessing(dist_params, key)
    log_prob = parametric_action_distribution.log_prob(dist_params, action)
    alpha = jnp.exp(log_alpha)
    alpha_loss = alpha * jax.lax.stop_gradient(-log_prob - target_entropy)

    loss = jnp.mean(alpha_loss)
    return loss


In [ ]:
%%writefile qdax/utils/metrics.py

"""Defines functions to retrieve metrics from training processes."""

from __future__ import annotations

import csv
from functools import partial
from typing import Dict, List

import jax
from jax import numpy as jnp

from qdax.core.containers.ga_repertoire import GARepertoire
from qdax.core.containers.mapelites_repertoire import MapElitesRepertoire
from qdax.core.containers.mome_repertoire import MOMERepertoire
from qdax.custom_types import Metrics
from qdax.utils.pareto_front import compute_hypervolume


class CSVLogger:
    """Logger to save metrics of an experiment in a csv file
    during the training process.
    """

    def __init__(self, filename: str, header: List) -> None:
        """Create the csv logger, create a file and write the
        header.

        Args:
            filename: path to which the file will be saved.
            header: header of the csv file.
        """
        self._filename = filename
        self._header = header
        with open(self._filename, "w") as file:
            writer = csv.DictWriter(file, fieldnames=self._header)
            # write the header
            writer.writeheader()

    def log(self, metrics: Dict[str, float]) -> None:
        """Log new metrics to the csv file.

        Args:
            metrics: A dictionary containing the metrics that
                need to be saved.
        """
        with open(self._filename, "a") as file:
            writer = csv.DictWriter(file, fieldnames=self._header)
            # write new metrics in a raw
            writer.writerow(metrics)


def default_ga_metrics(
    repertoire: GARepertoire,
) -> Metrics:
    """Compute the usual GA metrics that one can retrieve
    from a GA repertoire.

    Args:
        repertoire: a GA repertoire

    Returns:
        a dictionary containing the max fitness of the
            repertoire.
    """

    # get metrics
    max_fitness = jnp.max(repertoire.fitnesses, axis=0)

    return {
        "max_fitness": max_fitness,
    }


def default_qd_metrics(repertoire: MapElitesRepertoire, qd_offset: float) -> Metrics:
    """Compute the usual QD metrics that one can retrieve
    from a MAP Elites repertoire.

    Args:
        repertoire: a MAP-Elites repertoire
        qd_offset: an offset used to ensure that the QD score
            will be positive and increasing with the number
            of individuals.

    Returns:
        a dictionary containing the QD score (sum of fitnesses
            modified to be all positive), the max fitness of the
            repertoire, the coverage (number of niche filled in
            the repertoire).
    """

    # get metrics
    repertoire_empty = repertoire.fitnesses == -jnp.inf
    qd_score = jnp.sum(repertoire.fitnesses, where=~repertoire_empty)
    qd_score += qd_offset * jnp.sum(1.0 - repertoire_empty)
    coverage = 100 * jnp.mean(1.0 - repertoire_empty)
    max_fitness = jnp.max(repertoire.fitnesses)

    return {"qd_score": qd_score, "max_fitness": max_fitness, "coverage": coverage}


def default_qd_metrics_2(repertoire: MapElitesRepertoire, qd_offset: float, maxfit_offset: float) -> Metrics:
    """Compute the usual QD metrics that one can retrieve
    from a MAP Elites repertoire.

    Args:
        repertoire: a MAP-Elites repertoire
        qd_offset: an offset used to ensure that the QD score
            will be positive and increasing with the number
            of individuals.

    Returns:
        a dictionary containing the QD score (sum of fitnesses
            modified to be all positive), the max fitness of the
            repertoire, the coverage (number of niche filled in
            the repertoire).
    """

    # get metrics
    repertoire_empty = repertoire.fitnesses == -jnp.inf
    qd_score = jnp.sum(repertoire.fitnesses, where=~repertoire_empty)
    qd_score += qd_offset * jnp.sum(1.0 - repertoire_empty)
    coverage = 100 * jnp.mean(1.0 - repertoire_empty)
    max_fitness = jnp.max(repertoire.fitnesses) - maxfit_offset

    return {"qd_score": qd_score, "max_fitness": max_fitness, "coverage": coverage}


def default_moqd_metrics(
    repertoire: MOMERepertoire, reference_point: jnp.ndarray
) -> Metrics:
    """Compute the MOQD metric given a MOME repertoire and a reference point.

    Args:
        repertoire: a MOME repertoire.
        reference_point: the hypervolume of a pareto front has to be computed
            relatively to a reference point.

    Returns:
        A dictionary containing all the computed metrics.
    """
    repertoire_empty = repertoire.fitnesses == -jnp.inf
    repertoire_empty = jnp.all(repertoire_empty, axis=-1)
    repertoire_not_empty = ~repertoire_empty
    repertoire_not_empty = jnp.any(repertoire_not_empty, axis=-1)
    coverage = 100 * jnp.mean(repertoire_not_empty)
    hypervolume_function = partial(compute_hypervolume, reference_point=reference_point)
    moqd_scores = jax.vmap(hypervolume_function)(repertoire.fitnesses)
    moqd_scores = jnp.where(repertoire_not_empty, moqd_scores, -jnp.inf)
    max_hypervolume = jnp.max(moqd_scores)
    max_scores = jnp.max(repertoire.fitnesses, axis=(0, 1))
    max_sum_scores = jnp.max(jnp.sum(repertoire.fitnesses, axis=-1), axis=(0, 1))
    num_solutions = jnp.sum(~repertoire_empty)
    (
        pareto_front,
        _,
    ) = repertoire.compute_global_pareto_front()

    global_hypervolume = compute_hypervolume(
        pareto_front, reference_point=reference_point
    )
    metrics = {
        "moqd_score": moqd_scores,
        "max_hypervolume": max_hypervolume,
        "max_scores": max_scores,
        "max_sum_scores": max_sum_scores,
        "coverage": coverage,
        "number_solutions": num_solutions,
        "global_hypervolume": global_hypervolume,
    }

    return metrics


In [ ]:
from IPython.display import clear_output


print("Installing...")
!pip install -e .
!pip install absl-py==1.0.0
!pip install chex==0.1.86
!pip install flax==0.8.5
!pip install jumanji==0.3.1
!pip install optax==0.1.9
!pip install protobuf==3.19.5
!pip install scikit-learn==1.5.1
!pip install scipy==1.10.1
!pip install tensorflow-probability==0.24.0
!pip install typing-extensions==4.12.2
!pip install brax==0.10.4
!pip install jax[cuda12]==0.4.28
!pip install numpy==1.26.4

clear_output()

In [ ]:
!pip install ipympl | tail -n 1
# %matplotlib widget
# from google.colab import output
# output.enable_custom_widget_manager()

import os

from IPython.display import clear_output
import functools
import time
from typing import Any, Tuple

import jax
import jax.numpy as jnp
from brax.training.distribution import NormalTanhDistribution
import qdax

import qdax.tasks.brax.v1 as environments
from qdax.core.containers.mapelites_repertoire import compute_cvt_centroids
from qdax.core.emitters.dcrl_me_sac_emitter import DCRLMEConfig, DCRLMEEmitter
from qdax.core.emitters.mutation_operators import isoline_variation
from qdax.core.map_elites import MAPElites
from qdax.core.neuroevolution.buffers.buffer import DCRLTransition
from qdax.core.neuroevolution.networks.networks import MLP, MLPDC
from qdax.custom_types import EnvState, Params, RNGKey
from qdax.tasks.brax.v1 import descriptor_extractor
from qdax.tasks.brax.v1.wrappers.reward_wrappers import OffsetRewardWrapper, ClipRewardWrapper
from qdax.tasks.brax.v1.env_creators import scoring_function_brax_envs
from qdax.utils.plotting import plot_map_elites_results

from qdax.utils.metrics import CSVLogger, default_qd_metrics, default_qd_metrics_2


if "COLAB_TPU_ADDR" in os.environ:
  from jax.tools import colab_tpu
  colab_tpu.setup_tpu()

clear_output()

In [ ]:
#@title QD Training Definitions Fields
seed = 1 #@param {type:"integer"}

env_name = "ant_omni" #@param['ant_uni', 'hopper_uni', 'walker_uni', 'halfcheetah_uni', 'humanoid_uni', 'ant_omni', 'humanoid_omni']
episode_length = 250 #@param {type:"integer"}
min_descriptor = -30 #@param {type:"number"}
max_descriptor = 30 #@param {type:"number"}

num_iterations = 3907 #@param {type:"integer"}
batch_size = 256 #@param {type:"integer"}

# Archive
num_init_cvt_samples = 50000 #@param {type:"integer"}
num_centroids = 1024 #@param {type:"integer"}
policy_hidden_layer_sizes = (128, 128) #@param {type:"raw"}

# DCRL-ME
ga_batch_size = 128 #@param {type:"integer"}
dcrl_batch_size = 64 #@param {type:"integer"}
ai_batch_size = 64 #@param {type:"integer"}
lengthscale = 1 * 0.1 #@param {type:"number"}

# GA emitter
iso_sigma = 0.005 #@param {type:"number"}
line_sigma = 0.05 #@param {type:"number"}

# DCRL emitter
critic_hidden_layer_size = (256, 256) #@param {type:"raw"}
num_critic_training_steps = 3000 #@param {type:"integer"}
num_pg_training_steps = 150 #@param {type:"integer"}
replay_buffer_size = 1_000_000 #@param {type:"integer"}
discount = 0.99 #@param {type:"number"}
reward_scaling = 1.0 #@param {type:"number"}
critic_learning_rate = 3e-4 #@param {type:"number"}
actor_learning_rate = 3e-4 #@param {type:"number"}
policy_learning_rate = 5e-3 #@param {type:"number"}
alpha_learning_rate = 3e-4 #@param {type:"number"}
soft_tau_update = 0.005 #@param {type:"number"}
policy_delay = 1 #@param {type:"number"}
alpha_init = 1.0 #@param {type:"number"}
#@markdown ---

## Init environment, policy, population params, init states of the env

Define the environment in which the policies will be trained. In this notebook, we focus on controllers learning to move a robot in a physical simulation. We also define the shared policy, that every individual in the population will use. Once the policy is defined, all individuals are defined by their parameters, that corresponds to their genotype.

In [ ]:
# Init a random key
key = jax.random.key(seed)

# Init environment
env = environments.create(env_name, episode_length=episode_length)
env = OffsetRewardWrapper(
    env, offset=environments.reward_offset[env_name]
)  # apply reward offset as DCRL needs positive rewards
env = ClipRewardWrapper(
    env, clip_min=0.,
)  # apply reward clip as DCRL needs positive rewards
reset_fn = jax.jit(env.reset)

# Init policy network
policy_layer_sizes = policy_hidden_layer_sizes + (2 * env.action_size,)
policy_network = MLP(
    layer_sizes=policy_layer_sizes,
    kernel_init=jax.nn.initializers.lecun_uniform(),
    final_activation=None,
)
actor_dc_network = MLPDC(
    layer_sizes=policy_layer_sizes,
    kernel_init=jax.nn.initializers.lecun_uniform(),
    final_activation=None,
)

# Init population of controllers
key, subkey = jax.random.split(key)
keys = jax.random.split(subkey, num=batch_size)
fake_batch_obs = jnp.zeros(shape=(batch_size, env.observation_size))
init_params = jax.vmap(policy_network.init)(keys, fake_batch_obs)

sample_action_cls = NormalTanhDistribution(event_size=env.action_size)

## Define the way the policy interacts with the env

In [ ]:
# Define the function to play a step with the policy in the environment
def select_action(
    obs,
    policy_network,
    policy_params: Params,
    key: RNGKey,
    deterministic: bool = False,
):
    """Selects an action according to SAC policy.

    Args:
        obs: agent observation(s)
        policy_params: parameters of the agent's policy
        key: jax random key
        deterministic: whether to select action in a deterministic way.
            Defaults to False.

    Returns:
        The selected action and a new random key.
    """

    dist_params = policy_network.apply(policy_params, obs)
    if not deterministic:
        actions = sample_action_cls.sample(dist_params, key)
    

    else:
        # The first half of parameters is for mean and the second half for variance
        actions = jax.nn.tanh(dist_params[..., : dist_params.shape[-1] // 2])

    return actions

def play_step_fn(
    env_state: EnvState, policy_params: Params, key: RNGKey
) -> Tuple[EnvState, Params, RNGKey, DCRLTransition]:
    key, subkey = jax.random.split(key)
    actions = select_action(
        policy_network=policy_network,
        obs=env_state.obs,
        policy_params=policy_params,
        key=subkey,
    )
    state_desc = env_state.info["state_descriptor"]
    next_state = env.step(env_state, actions)

    transition = DCRLTransition(
        obs=env_state.obs,
        next_obs=next_state.obs,
        rewards=next_state.reward,
        dones=next_state.done,
        truncations=next_state.info["truncation"],
        actions=actions,
        state_desc=state_desc,
        next_state_desc=next_state.info["state_descriptor"],
        desc=jnp.zeros(
            env.descriptor_length,
        )
        * jnp.nan,
        desc_prime=jnp.zeros(
            env.descriptor_length,
        )
        * jnp.nan,
    )

    return next_state, policy_params, key, transition

## Define the scoring function and the way metrics are computed

The scoring function is used in the evaluation step to determine the fitness and behavior descriptor of each individual.

In [ ]:
# Prepare the scoring function
descriptor_extraction_fn = descriptor_extractor[env_name]
scoring_fn = functools.partial(
    scoring_function_brax_envs,
    episode_length=episode_length,
    play_reset_fn=reset_fn,
    play_step_fn=play_step_fn,
    descriptor_extractor=descriptor_extraction_fn,
)

# Get minimum reward value to make sure qd_score are positive
reward_offset = environments.reward_offset[env_name]
print(reward_offset)

# Define a metrics function
metrics_function = functools.partial(
    default_qd_metrics,
    qd_offset=reward_offset * episode_length,
)   # The env wrapper has already offseted the reward

## Define the emitter: DCRL Emitter

The emitter is used to evolve the population at each mutation step. In this example, the emitter is the Descriptor-Conditioned RL emitter, the one used in DCRL-ME. It trains a critic with the transitions experienced in the environment and uses the critic to apply Descriptor-Conditioned gradients updates to the policies evolved.

In [ ]:
dcrl_emitter_config = DCRLMEConfig(
    ga_batch_size=ga_batch_size,
    dcrl_batch_size=dcrl_batch_size,
    ai_batch_size=ai_batch_size,
    lengthscale=lengthscale,
    critic_hidden_layer_size=critic_hidden_layer_size,
    num_critic_training_steps=num_critic_training_steps,
    num_pg_training_steps=num_pg_training_steps,
    batch_size=batch_size,
    replay_buffer_size=replay_buffer_size,
    discount=discount,
    reward_scaling=reward_scaling,
    critic_learning_rate=critic_learning_rate,
    actor_learning_rate=actor_learning_rate,
    policy_learning_rate=policy_learning_rate,
    alpha_learning_rate=alpha_learning_rate,
    soft_tau_update=soft_tau_update,
    policy_delay=policy_delay,
    alpha_init=alpha_init,
)

In [ ]:
# Get the emitter
variation_fn = functools.partial(
    isoline_variation, iso_sigma=iso_sigma, line_sigma=line_sigma
)

dcrl_emitter = DCRLMEEmitter(
    config=dcrl_emitter_config,
    policy_network=policy_network,
    actor_network=actor_dc_network,
    env=env,
    variation_fn=variation_fn,
)

## Instantiate and initialise the MAP Elites algorithm

In [ ]:
# Instantiate MAP Elites
map_elites = MAPElites(
    scoring_function=scoring_fn,
    emitter=dcrl_emitter,
    metrics_function=metrics_function,
)

# Compute the centroids
key, subkey = jax.random.split(key)
centroids = compute_cvt_centroids(
    num_descriptors=env.descriptor_length,
    num_init_cvt_samples=num_init_cvt_samples,
    num_centroids=num_centroids,
    minval=min_descriptor,
    maxval=max_descriptor,
    key=subkey,
)

# compute initial repertoire
key, subkey = jax.random.split(key)
repertoire, emitter_state, init_metrics = map_elites.init(
    init_params, centroids, subkey
)

In [ ]:
from jax.flatten_util import ravel_pytree

def save(repertoire, path: str = "./", i: int = 0) -> None:
        """Saves the repertoire on disk in the form of .npy files.

        Flattens the genotypes to store it with .npy format. Supposes that
        a user will have access to the reconstruction function when loading
        the genotypes.

        Args:
            path: Path where the data will be saved. Defaults to "./".
        """

        def flatten_genotype(genotype) -> jnp.ndarray:
            flatten_genotype, _ = ravel_pytree(genotype)
            return flatten_genotype

        # flatten all the genotypes
        flat_genotypes = jax.vmap(flatten_genotype)(repertoire.genotypes)

        # save data
        jnp.save(path + f"genotypes-{i}.npy", flat_genotypes)
        jnp.save(path + f"fitnesses-{i}.npy", repertoire.fitnesses)
        jnp.save(path + f"descriptors-{i}.npy", repertoire.descriptors)
        jnp.save(path + f"centroids-{i}.npy", repertoire.centroids)
        jnp.save(path + f"extra_scores-{i}.npy", repertoire.extra_scores)
        jnp.save(path + f"keys_extra_scores-{i}.npy", repertoire.keys_extra_scores)

In [ ]:
from tqdm import tqdm
log_period = 10
num_loops = num_iterations // log_period

# Initialize metrics
metrics = {key: jnp.array([]) for key in ["iteration", "qd_score", "coverage", "max_fitness", "time"]}

# Set up init metrics
init_metrics = jax.tree.map(lambda x: jnp.array([x]) if x.shape == () else x, init_metrics)
init_metrics["iteration"] = jnp.array([0], dtype=jnp.int32)
init_metrics["time"] = jnp.array([0.0])  # No time recorded for initialization

# Convert init_metrics to match the metrics dictionary structure
metrics = jax.tree.map(lambda metric, init_metric: jnp.concatenate([metric, init_metric], axis=0), metrics, init_metrics)

# Initialize CSV logger
csv_logger = CSVLogger(
    "dcrlme-sac-logs.csv",
    header=list(metrics.keys())
)

save(repertoire, "./", -1)

# Main loop
map_elites_scan_update = map_elites.scan_update
for i in tqdm(range(num_loops)):
    start_time = time.time()
    (
        repertoire,
        emitter_state,
        key,
    ), current_metrics = jax.lax.scan(
        map_elites_scan_update,
        (repertoire, emitter_state, key),
        (),
        length=log_period,
    )
    timelapse = time.time() - start_time

    # Metrics
    current_metrics["iteration"] = jnp.arange(1+log_period*i, 1+log_period*(i+1), dtype=jnp.int32)
    current_metrics["time"] = jnp.repeat(timelapse, log_period)
    metrics = jax.tree.map(lambda metric, current_metric: jnp.concatenate([metric, current_metric], axis=0), metrics, current_metrics)

    # Log
    csv_logger.log(jax.tree.map(lambda x: x[-1], metrics))
    if i in [1, 4, 7, 10, 15, 20, 35, 50, 75, 100, 200, 300, 389, 390]:
        save(repertoire, "./", i)

In [ ]:
#@title Visualization

# Create the x-axis array
env_steps = metrics["iteration"]

%matplotlib inline
# Create the plots and the grid
fig, axes = plot_map_elites_results(env_steps=env_steps, metrics=metrics, repertoire=repertoire, min_descriptor=min_descriptor, max_descriptor=max_descriptor)